# 4 · The hardware experiment: setup, budget and circuits

This notebook turns notebooks 1–3 into a concrete Tuna-17 run:
- the choice of size, depth, ς and SPSA budget;
- what it costs in submissions and hours;
- the exact circuits that will be submitted;
- the run plan, written to `tfd_study/hardware_plan.json` and executed by `tfd_study/run_hardware.py`. That runner uses the existing scheduler of `gibbs_tfd_reproduction/run_hardware_parallel.py`, unchanged.

**Budget reference.** The last parallel Tuna-17 run (`results/parallel_20261019_175824`) completed 603 submissions in 18.3 h, about 33 per hour. Quantum Inspire allows 5 circuits per submission and 5 submissions in the queue. "One night" ≈ 12–16 h ≈ 400–530 submissions.

In [1]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")          # before numpy: see tfd_study/__init__.py
import json, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "tfd_study").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
from tfd_study import model, devices
from tfd_study.plotting import *
apply_style()
RES = ROOT / "tfd_study" / "results"
FIGS = ROOT / "tfd_study" / "figures"; FIGS.mkdir(exist_ok=True)
pd.set_option("display.precision", 4)

import math
from tfd_study.run_hardware import budget, SUBMISSIONS_PER_HOUR

## Candidate setups and what they cost

In [2]:
chains = [json.loads(l) for l in (RES / "spsa_study.jsonl").read_text(encoding="utf-8").splitlines()]
rows = []
for c in chains:
    if c["part"] != "main" or c["kind"] != "noisy":
        continue
    for st in c["steps"]:
        rows.append({"n": c["n"], "p": c["p"], "s": c["s"], "iters": c["iters"], "seed": c["seed"],
                     "beta": st["beta"], "F": st["fid_device"], "F_angles": st["fid_angles"]})
sim = pd.DataFrame(rows)
cands = []
for (n, p, it), g in sim.groupby(["n", "p", "iters"]):
    for replicas in (2, 4):
        cfg = {"n": n, "p": p, "betas": [0.5, 1.0, 2.0, 5.0], "replicas": replicas, "iters": it, "calibrate": 3,
               "n_kicks": 8, "max_circuits": 5, "reference_end": True, "cal_interval": 30}
        b = budget(cfg)
        cands.append({"n": n, "p": p, "ς": g.s.iloc[0], "SPSA steps": it, "replicas": replicas,
                      "sim F (mean over β)": g.F.mean(), "sim F worst β": g.groupby("beta").F.mean().min(),
                      "submissions": b["total"], "hours": b["hours"]})
cand = pd.DataFrame(cands)
night = cand[cand.hours <= 16].sort_values(["n", "sim F (mean over β)"], ascending=[True, False])
night.groupby("n").head(3).set_index(["n", "p", "SPSA steps", "replicas"]).round(3)

ς  sim F (mean over β)  sim F worst β  \
n p SPSA steps replicas                                             
2 1 40         2         1.50                0.865          0.722   
    20         2         1.50                0.841          0.716   
               4         1.50                0.841          0.716   
3 1 40         2         1.57                0.748          0.601   
    20         2         1.57                0.713          0.583   
  2 40         2         1.30                0.661          0.427   
4 1 20         2         1.57                0.630          0.419   
  2 20         2         1.30                0.543          0.301   
  3 20         2         1.30                0.438          0.157   

                         submissions  hours  
n p SPSA steps replicas                      
2 1 40         2                 409   12.4  
    20         2                 239    7.2  
               4                 460   13.9  
3 1 40         2                 477   14.5  
    20         2                 307    9.3  
  2 40         2                 477   14.5  
4 1 20         2                 494   15.0  
  2 20         2                 494   15.0  
  3 20         2                 494   15.0

## Decision: 3+3 qubits, one layer, SPSA 40 steps, 2 replicas

| Choice | Value | Why |
|---|---|---|
| size | **3+3** (6 qubits) | The first size beyond the paper's 2+2, which we have already run on Tuna-17. 4+4 is affordable only as a single replica, with half the budget spent on 81-setting tomography and four never-characterized qubits. |
| depth | **p = 1** (17 CZ) | Best at every size and every noise variant (notebooks 2–3). p = 2 gains about 0.02 without noise but loses about 0.10 on the device. |
| ς | **1.57** | The re-fitted best for one layer (1.5–1.57 is flat). Under noise ς barely matters anyway. |
| layout | **Tuna-17 qubits 2-4-7-5-8-11** (A1 A2 A3 B1 B2 B3) | A ladder with no SWAPs. It reuses qubits 2, 4, 5 and 8 from our 2+2 runs, on which the noise model was fitted. |
| β | **0.5, 1, 2, 5** | Same grid as the study. β = 5 is the hardest point (sim 0.60) but shows the low-temperature trend. |
| SPSA | **40 steps per β**, gain calibration 3 pairs, first step 0.3 rad, 8 random kicks at the first β | Within 0.035 of the noisy ceiling (3+3: 0.748 vs 0.783). 80 steps would gain 0.02 but double the cost. |
| replicas | **2** | The minimum to see run-to-run spread (simulated std 0.033 at 40 steps). |
| tomography | 27 settings × 4096 shots, ρ_A and ρ_B; the reference angles both at start and at end | The start references are known circuits on the exact qubits, so they check the noise model. The end references measure drift. |
| shots | 4096 per loss circuit | As simulated. |

**Expected on Tuna-17** (simulation with the fitted model, mean of 4 seeds): fidelity ≈ 0.84 / 0.84 / 0.72 / 0.60 at β = 0.5 / 1 / 2 / 5. The model was fitted on 2+2 data and is slightly optimistic at small β, so a few percent lower would still be consistent.

**Budget:** about 477 submissions (344 SPSA, 8 kicks, 96 tomography, ~30 calibrations), about 14.5 h at the 33 submissions/h of the last run. Queue waits dominate, so the real duration depends on how busy Tuna-17 is.

In [3]:
plan = json.loads((ROOT / "tfd_study" / "hardware_plan.json").read_text(encoding="utf-8"))
run = plan["run"]
pd.Series({k: v for k, v in run.items() if k != "reference_angles"}, name="hardware_plan.json").to_frame()

,hardware_plan.json
backend,Tuna-17
n,3
p,1
varsigma,1.57
betas,"[0.5, 1.0, 2.0, 5.0]"
layout,"[2, 4, 7, 5, 8, 11]"
replicas,2
iters,40
calibrate,3
first_step,0.3


In [4]:
cfg = {**run, "max_circuits": 5}
b = budget(cfg)
pd.Series(b, name="estimated").to_frame()

,estimated
cost,344.0
kicks,8.0
tomography,96.0
calibrations,29.0
total,477.0
hours,14.5


## The circuits that will run

One SPSA loss evaluation is two circuits per parameter point: all-Z and all-X readout. Below is the all-X circuit at the noiseless-optimal angles for β = 1, compiled to Tuna-17's native gates (`rx`, `ry`, `rz`, `cz`) on the chosen physical qubits. All circuits of the run, compiled, are written to `tfd_study/hardware_circuits/` as OpenQASM 3: the loss circuits, every tomography setting and the readout calibration.

In [5]:
from qiskit import qasm3
from tfd_study.run_hardware import reference_angles
n, p, s = run["n"], run["p"], run["varsigma"]
ref = reference_angles(n, p, s, run["betas"])
th = ref["1.0"]
qc = devices.to_device(model.measured(n, p, th, "X"), run["layout"])
print(f"compiled: {qc.count_ops().get('cz', 0)} CZ, depth {qc.depth()}, on Tuna-17 qubits {run['layout']}")
qc.draw("mpl", fold=40, idle_wires=False, style="clifford")

compiled: 17 CZ, depth 22, on Tuna-17 qubits [2, 4, 7, 5, 8, 11]


<Figure size 2866.54x618.722 with 1 Axes>

In [6]:
out = ROOT / "tfd_study" / "hardware_circuits"; out.mkdir(exist_ok=True)
for f in out.glob("*.qasm"): f.unlink()
files = {}
for b in run["betas"]:
    th_b = ref[str(b)]
    for basis in ("Z", "X"):
        files[f"loss_beta{b:g}_{basis}.qasm"] = model.measured(n, p, th_b, basis)
    for setting in model.tomography_settings(n):
        files[f"tomo_beta{b:g}_{setting[:n]}.qasm"] = model.measured(n, p, th_b, setting)
for i, c in enumerate(model.calibration_circuits(n)):
    files[f"readout_cal{i}.qasm"] = c
for name, c in files.items():
    (out / name).write_text(qasm3.dumps(devices.to_device(c, run["layout"])))
print(f"{len(files)} compiled circuits written to {out.relative_to(ROOT)}")

118 compiled circuits written to tfd_study\hardware_circuits


## Running it

From the repository root:

```bash
python -m tfd_study.run_hardware --config tfd_study/hardware_plan.json --self-test   # ideal sim: checks bit order, bases, readout inversion, tomography
python -m tfd_study.run_hardware --config tfd_study/hardware_plan.json --dry-run     # whole protocol on the fitted noise model, tiny budget
qi login                                                                              # token must be fresh for a long run
python -m tfd_study.run_hardware --config tfd_study/hardware_plan.json               # the real run
python -m tfd_study.run_hardware --resume results/tfd_n<n>p<p>_<timestamp>            # after an interruption
```

Every job's counts and id are appended to `raw_jobs.jsonl`, and `state.json` is checkpointed after every submission. The reference tomographies at the start use known circuits on the exact qubits of the run. Compare them with the noise model's prediction for the same angles (the noisy ceiling in notebook 2) before interpreting the replicas. That tells whether the qubits of this layout behave like the 2-4-5-8 data the model was fitted to.